## Carregamento de dataset de exemplo

Para exemplicação na disciplina, iremos trabalhar com o paradigma P300, utilizando o P300-Dataset do site [1].

Este dataset possui 8 participantes, e será carregado somente a base de dados de um deles para seguir com os exemplos.

A aplicação é de um speller (soletrador) 6x6, o qual foi utilizado nas seções experimentais, 7 palavras com 5 letras cada. Das 7 palavras, 3 foram utilizadas para treino e 4 para teste.

O equipamento utilizado foi o "g.Tec g.Nautilus g.LadyBird", com taxa de amostragem de 250 Hz, filtro `notch` de 50Hz (europeu) e filtro `bandpass` de 0.1 à 30 Hz.



In [35]:
# Carregamento do P300-dataset para o participante 8

from scipy.io import loadmat
import numpy as np

raw_data = loadmat('P300S08.mat')
raw_data.keys()

dict_keys(['__header__', '__version__', '__globals__', 'channelRange', 'Fs', 'Trials', 'stims', 'channelNames', 'sampleTime', 'samples', 'Word', 'data', 'subject'])

In [36]:
# Apresentando informações básicas presentes no dataset

# Taxa de amostragem
fs = raw_data['Fs'][0][0]
print(f'Taxa de amostragem: {fs} Hz')

# Canais utilizados
channels = raw_data['channelNames']
print('\nCanais utilizados: ', [str(name[0]) for name in channels[0]])
print('Número de canais: ', len(channels[0]))

Taxa de amostragem: 250 Hz

Canais utilizados:  ['FZ', 'CZ', 'P3', 'PZ', 'P4', 'PO7', 'PO8', 'OZ']
Número de canais:  8


In [37]:
# preparando os dados principais

main_data = raw_data['data'].copy()

X = main_data['X'][0][0]
y = main_data['y'][0][0].T[0]
stim = main_data['y_stim'][0][0].T[0]
trials = main_data['trial'][0][0][0]
flash = main_data['flash'][0][0]

# dados flutuantes / canais
print(X.shape)

# rótulos
print(y.shape)

# aparentemente, matriz com 0's contabilizando os estímulos
print(stim.shape)

# momento da amostragem onde cada um dos 35 ensaios começam
print(trials.shape)

# momento da amostragem onde cada flash é executado
# (ID da amostra, duração, estímulo, acerto/erro)
print(flash.shape)

(355904, 8)
(355904,)
(355904,)
(35,)
(4200, 4)


In [38]:
print(trials)
print(flash, len(flash))

[  7478  17505  27533  37558  47580  57592  67628  77657  87682  97707
 107734 117759 127784 137808 147834 157860 167888 177910 187953 197955
 207987 218012 228037 238064 248088 258114 268138 278167 288194 298219
 308239 318275 328281 338317 348341]
[[  7478     32      7      1]
 [  7541     32      3      1]
 [  7604     32     10      1]
 ...
 [355682     31      3      2]
 [355745     31     10      1]
 [355807     31      4      1]] 4200


### Organização das informações da base de dados

Uma vez carregada e separada em variáveis distintas, é necessário agrupar os valores em apenas uma estrutura de dados (X) de forma consistente.

As próximas fases descrevem isto.

In [39]:
# separando os dados em blocos de flash

data_f = list()

for nf in range(len(flash)):
    if nf < len(flash) - 1:
        data_f.append(X[flash[nf][0]:flash[nf + 1][0], :])
data_f.append(X[flash[nf][0], :])


In [40]:
# separando os dados em blocos de ensaio (trials)

data_t = list()

c1 = 0
c2 = 0
for f in flash:
    if f[3] == 1:
        c1 += 1
    elif f[3] == 2:
        c2 += 1

print(c1, c2)

3500 700


In [41]:
# Este código ajuda a entender qual é a quantidade de dados em cada flash, ou seja, quantas 
# amostras de dados existem para cada flash. Isso é útil para verificar a consistência dos dados 
# e garantir que cada flash tenha a quantidade esperada de amostras.

chs = dict()

for s in set([len(f) for f in data_f]):
    chs[s] = 0

for s in [len(f) for f in data_f]:
    chs[s] += 1

for k, v in chs.items():
    print(f'{k}: {v}', end=' | ')


0: 302 | 2561: 2 | 2560: 2 | 2563: 2 | 2564: 2 | 2565: 5 | 2566: 3 | 2567: 1 | 2562: 2 | 2569: 2 | 2570: 1 | 2568: 1 | 8: 1 | 2573: 1 | 2577: 1 | 2578: 1 | 2583: 1 | 36: 1 | 37: 1 | 38: 7 | 39: 7 | 40: 30 | 41: 25 | 42: 16 | 43: 1 | 52: 3 | 53: 18 | 54: 24 | 55: 34 | 56: 46 | 57: 55 | 58: 24 | 59: 155 | 60: 366 | 61: 341 | 62: 537 | 63: 1014 | 64: 526 | 65: 25 | 66: 4 | 67: 3 | 68: 9 | 69: 30 | 70: 23 | 71: 32 | 72: 55 | 73: 23 | 74: 2 | 2626: 1 | 76: 1 | 2630: 1 | 83: 3 | 84: 3 | 85: 6 | 86: 8 | 87: 13 | 88: 16 | 89: 29 | 90: 28 | 91: 10 | 92: 6 | 93: 1 | 94: 1 | 96: 1 | 97: 1 | 99: 1 | 101: 2 | 103: 1 | 105: 3 | 113: 2 | 116: 5 | 117: 4 | 118: 3 | 119: 10 | 120: 10 | 121: 4 | 122: 15 | 123: 37 | 124: 40 | 125: 46 | 126: 30 | 127: 46 | 128: 8 | 129: 2 | 130: 4 | 131: 2 | 132: 3 | 134: 3 | 135: 2 | 148: 2 | 149: 1 | 150: 4 | 151: 3 | 152: 2 | 154: 2 | 155: 1 | 161: 1 | 2623: 1 | 2624: 1 | 2550: 1 | 2554: 1 | 2558: 1 | 

Como nem todos os índices de "trials" estão contidos em índices de "flash", os dados de flash (4200) serão divididos igualmente pela quantidade de trials (35), obtendo exatamente 120 conjuntos de intensificação.

Mas antes disso, vamos padronizar a quantidade de flashes para que futuramente possamos criar uma matriz perdeita utilizando `numpy.array`. Verificamos que o *trial* que mais aparece é o de tamanho **64**. Desta forma, iremos padronizar todos para este número. Quando houver tamanho zero, tudo será completado com valor **0**.

In [42]:
data_f_padronizado = []

for segmento in data_f:
    segmento = np.asarray(segmento)

    # Garante o formato (amostras, canais), inclusive para uma única amostra
    if segmento.size > 0 and segmento.ndim == 1:
        segmento = segmento.reshape(1, -1)

    if segmento.size == 0:
        segmento = np.zeros((n_amostras, n_canais), dtype=X.dtype)
    elif len(segmento) < n_amostras:
        repeticoes = int(np.ceil(n_amostras / len(segmento)))
        segmento = np.tile(segmento, (repeticoes, 1))[:n_amostras]
    else:
        segmento = segmento[:n_amostras]

    data_f_padronizado.append(segmento)

data_f = np.asarray(data_f_padronizado)
print("Formato de data_f:", data_f.shape)

Formato de data_f: (4200, 64, 8)


In [43]:
# salvamento dos dados em numpy arrays

# data = np.array(data_t)
# np.save('data_f.npy', data_f)